# Diabetic Retinopathy — Training Notebook (Colab)

**Workflow:**
1. Mount Google Drive
2. Set `STRATEGY` (and `BACKBONE` if fine-tuning)
3. Run all cells — the notebook trains the model, saves a named checkpoint to Drive, and generates a ready-to-submit ZIP.

**Drive layout expected:**
```
MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/
  data/          ← images + train/val/test CSVs
  src/           ← Python modules (copy of repo src/)
  models/
    custom/      ← one .pth per completed run, e.g. custom_auc0.742_20260416_143022.pth
    fine_tune/   ← e.g. fine_tune_alexnet_auc0.821_20260416_150311.pth
  results/
    custom_results/      ← one .csv per completed run
    fine_tune_results/   ← one .csv per completed run
    Submissions/         ← codabench_submission.zip (always up to date)
```

**Naming convention:** `{strategy}[_{backbone}]_auc{val_auc}_{YYYYMMDD_HHMMSS}`

## 1. Mount Drive & set up paths

In [23]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import sys
import os

DRIVE_BASE   = '/content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy'
DATA_DIR     = os.path.join(DRIVE_BASE, 'data')
# Updated to the correct path
SRC_DIR      = os.path.join(DRIVE_BASE, 'src')
MODELS_DIR   = os.path.join(DRIVE_BASE, 'models')
RESULTS_DIR  = os.path.join(DRIVE_BASE, 'results')

# Make both the base project directory and the src directory importable
for path in [DRIVE_BASE, SRC_DIR]:
    if path not in sys.path:
        sys.path.insert(0, path)

# Create output directories
for d in [
    os.path.join(MODELS_DIR,  'custom'),
    os.path.join(MODELS_DIR,  'fine_tune'),
    os.path.join(RESULTS_DIR, 'custom_results'),
    os.path.join(RESULTS_DIR, 'fine_tune_results'),
    os.path.join(RESULTS_DIR, 'Submissions'),
]:
    os.makedirs(d, exist_ok=True)

print(f'Paths updated. SRC_DIR set to: {SRC_DIR}')

Paths updated. SRC_DIR set to: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src


### 1.1 Speed up Training: Copy Data to Local Storage
We copy the dataset from Drive to the local Colab runtime to avoid slow network I/O during training.

In [25]:
import shutil
import os

# Define local path
LOCAL_DATA_DIR = '/content/data'

if not os.path.exists(LOCAL_DATA_DIR):
    print("Copying data to local storage...")
    # Assuming images are in a folder or zip. Adjust if you have a .zip file specifically.
    # If it's a folder of images:
    shutil.copytree(DATA_DIR, LOCAL_DATA_DIR)
    print("Data copied to local storage.")

# Update DATA_DIR to point to local disk
DATA_DIR = LOCAL_DATA_DIR
print(f"Updated DATA_DIR to: {DATA_DIR}")

Updated DATA_DIR to: /content/data


In [26]:
import os
# Diagnostic to find any .py files in the new SRC_DIR
print(f"Buscando archivos de código en: {SRC_DIR}")

if os.path.exists(SRC_DIR):
    found_code = False
    for root, dirs, files in os.walk(SRC_DIR):
        for file in files:
            if file.endswith('.py'):
                print(f"Encontrado: {os.path.join(root, file)}")
                found_code = True

    if not found_code:
        print("No se encontraron archivos .py en SRC_DIR. ¿Estás seguro de que esta es la carpeta correcta?")
else:
    print(f"La ruta SRC_DIR no es válida: {SRC_DIR}")

Buscando archivos de código en: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/__init__.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/data/transforms.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/data/dataset.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/data/__init__.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/model/__init__.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/model/custom_net.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src/model/fine_tune_net.py
Encontrado: /content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinop

## 2. Install dependencies *(run once if needed)*

In [27]:
# Uncomment on first run in a fresh Colab session
# !pip install -q scikit-image opencv-python-headless scikit-learn

## 3. Imports

In [28]:
import random
import shutil
from datetime import datetime
import sys, os

SRC_DIR = '/content/drive/MyDrive/Computer Vision/AutomaticDiagnosisForDiabeticRetinopathy/src'
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
if 'data' in sys.modules:
    del sys.modules['data']

import numpy as np
import numpy.random as npr
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

try:
    from data.dataset        import RetinopathyDataset
    from data.transforms     import get_train_transforms, get_val_transforms
    from model.ensemble_net  import DenseNetGreen, ResNet9, EnsembleNet
    from model.fine_tune_net import build_fine_tune_model
    from training.trainer    import train_model
    from training            import config
    from evaluation.submission import (
        test_model, save_strategy_results, generate_submission
    )
    print('All imports OK.')
except ImportError as e:
    print(f'Import failed: {e}')


All imports OK.


## 4. Configuration

**Edit this cell** before each training run.

In [ ]:
# ── Which base model(s) to train ─────────────────────────────────────────────
# 'seresnet9'  →  SEResNet9 (SE-gated ResNet-9 backbone, recommended)
BASE_MODEL = 'seresnet9'

# Fine-tune backbone (ignored when training custom base models)
BACKBONE = 'alexnet'

# 0 = full dataset; set to a small int (e.g. 200) for fast debug runs
MAX_TRAIN_SIZE = 0

STRATEGY = 'custom'  # submission track — do not change

import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device     : {device}')
print(f'Base model : {BASE_MODEL}')
print(f'AMP        : {config.use_amp and device.type == "cuda"}')


## 5. Data loading

In [ ]:
import random
import numpy.random as npr
import torch
import os
import pandas as pd
from torch.utils.data import DataLoader, WeightedRandomSampler

from data.dataset import RetinopathyDataset
from data.transforms import get_train_transforms, get_val_transforms
from training import config

BATCH_SIZE     = config.ensemble_batch_size if STRATEGY == 'custom' else config.batch_size
VAL_BATCH      = config.ensemble_val_batch  if STRATEGY == 'custom' else 256
NUM_WORKERS    = config.ensemble_num_workers if STRATEGY == 'custom' else 2

random.seed(42)
npr.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.enabled = False

train_transform = get_train_transforms(img_size=config.img_height)
val_transform   = get_val_transforms(img_size=config.img_height)

train_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_DIR, 'train.csv'),
    root_dir=DATA_DIR,
    transform=train_transform,
    maxSize=MAX_TRAIN_SIZE,
)
val_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_DIR, 'val.csv'),
    root_dir=DATA_DIR,
    transform=val_transform,
)
test_dataset = RetinopathyDataset(
    csv_file=os.path.join(DATA_DIR, 'test.csv'),
    root_dir=DATA_DIR,
    transform=val_transform,
)

# Build per-sample weights inversely proportional to class frequency.
# This guarantees ~50/50 positive/negative in every batch.
# Read labels directly from CSV (avoids decoding all images just to get labels).
_df     = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'),
                      dtype={'id': str, 'eye': int, 'label': int})
_labels = ((_df['label'] > 0).astype(int)).tolist()
if MAX_TRAIN_SIZE > 0:
    _idx    = npr.RandomState(seed=42).permutation(range(len(_df)))
    _labels = [_labels[i] for i in _idx[:MAX_TRAIN_SIZE]]
_n_pos  = sum(_labels)
_n_neg  = len(_labels) - _n_pos
_weight_map = {0: 1.0 / _n_neg, 1: 1.0 / _n_pos}
_sample_weights = [_weight_map[l] for l in _labels]

_sampler = WeightedRandomSampler(
    weights=_sample_weights,
    num_samples=len(_sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=_sampler,          # sampler is mutually exclusive with shuffle=True
    num_workers=NUM_WORKERS,
    pin_memory=True,
)
print(f'Train sampler: {_n_pos} pos / {_n_neg} neg -> balanced batches')

val_loader   = DataLoader(val_dataset,   batch_size=VAL_BATCH,  shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=VAL_BATCH,  shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)

print(f'Using LOCAL data at: {DATA_DIR}')
print(f'Image size  : {config.img_height}×{config.img_width}')
print(f'Train batch : {BATCH_SIZE}  |  Val/Test batch: {VAL_BATCH}  |  Workers: {NUM_WORKERS}')
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')


## 6. Build model

In [ ]:
from model.ensemble_net  import SEResNet9
from training.losses     import FocalLoss

def build_base_model(name):
    """Instantiate SEResNet9, FocalLoss, Adam, SequentialLR (warmup + cosine)."""
    use_amp = config.use_amp and device.type == 'cuda'

    model = SEResNet9(in_channels=3, dropout=0.1).to(device)

    # FocalLoss with alpha=0.5: balanced batches (WeightedRandomSampler) remove
    # the need for alpha-based class reweighting; gamma=2.0 handles sample difficulty.
    criterion = FocalLoss(gamma=2.0, alpha=0.5)

    optimizer = optim.Adam(
        model.parameters(),
        lr=config.base_lr,
        weight_decay=config.base_weight_decay,
    )

    # LinearLR warmup (5 epochs: lr*0.01 -> lr) then CosineAnnealing for the rest
    warmup_sched = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=config.warmup_start_factor,
        end_factor=1.0,
        total_iters=config.warmup_epochs,
    )
    cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=config.base_num_epochs - config.warmup_epochs,
        eta_min=config.base_cosine_etamin,
    )
    scheduler = torch.optim.lr_scheduler.SequentialLR(
        optimizer,
        schedulers=[warmup_sched, cosine_sched],
        milestones=[config.warmup_epochs],
    )

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  SEResNet9: {n_params:,} params  |  FocalLoss(gamma=2.0, alpha=0.5)'
          f'  |  AMP={use_amp}')
    return model, criterion, optimizer, scheduler, config.base_num_epochs, use_amp


# Always train SEResNet9
models_to_train = ['seresnet9']
print(f'Will train: {models_to_train}')
for name in models_to_train:
    build_base_model(name)  # dry-run to catch config errors early


## 7. Train

The running best is kept at `_temp_best.pth` during training.  
Once done, it is renamed to a permanent file: `{strategy}[_{backbone}]_auc{auc}_{timestamp}.pth`.

In [32]:
def save_val_predictions(model, loader, device, model_name, models_dir):
    """Run inference on val set and save logits as .npy for later stacking."""
    model.eval()
    preds = []
    with torch.no_grad():
        for sample in loader:
            inputs = sample['image'].to(device).float()
            preds.append(torch.sigmoid(model(inputs)).cpu().numpy())
    preds = np.concatenate(preds, axis=0)  # (N_val, 1)
    out_path = os.path.join(models_dir, 'custom', f'{model_name}_val_preds.npy')
    np.save(out_path, preds)
    print(f'  Val predictions saved -> {out_path}')
    return preds


def run_training(model_name):
    random.seed(42); npr.seed(42)
    torch.manual_seed(42); torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

    model, criterion, optimizer, scheduler, num_epochs, use_amp = build_base_model(model_name)
    temp_path = os.path.join(MODELS_DIR, 'custom', '_temp_best.pth')

    print(f'\n=== Training {model_name} ===')
    model, best_auc = train_model(
        model=model, criterion=criterion, optimizer=optimizer,
        scheduler=scheduler,
        dataloaders={'train': train_loader, 'val': val_loader},
        dataset_sizes={'train': len(train_dataset), 'val': len(val_dataset)},
        device=device, num_epochs=num_epochs,
        temp_save_path=temp_path, use_amp=use_amp,
    )

    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    run_name  = f'{model_name}_auc{best_auc:.4f}_{timestamp}'
    final_path = os.path.join(MODELS_DIR, 'custom', f'{run_name}.pth')
    if os.path.exists(temp_path):
        shutil.move(temp_path, final_path)
    else:
        torch.save(model.state_dict(), final_path)
    print(f'  Model saved -> {final_path}')

    save_val_predictions(model, val_loader, device, model_name, MODELS_DIR)

    return model, best_auc, run_name


# ── Run ──────────────────────────────────────────────────────────────────────
trained = {}
for name in models_to_train:
    trained[name] = run_training(name)

# Expose last trained model and run name for the submission cell
last_name = models_to_train[-1]
model, best_auc, RUN_NAME = trained[last_name]
print(f'\nDone. Last run: {RUN_NAME}')


Epoch 0/49
----------
  train  loss: 0.6170  AUC: 0.5035
  val    loss: 0.5678  AUC: 0.5492
  -> New best (AUC=0.5492) saved to temp checkpoint

Epoch 1/49
----------


KeyboardInterrupt: 

## 8. Generate submission

- Runs inference on the test set.
- Saves scores as `{run_name}.csv` in `results/{strategy}_results/`.
- Looks for the other strategy's most recent CSV; uses **random scores** if none found.
- Creates `results/Submissions/codabench_submission.zip`.

In [ ]:
# Run test inference
outputs = test_model(model, test_loader, device)
print(f'Test outputs shape: {outputs.shape}')   # expected (1000, 1)

# Save this run's scores with the same descriptive name as the model
current_csv_path = save_strategy_results(outputs, STRATEGY, RESULTS_DIR, RUN_NAME)

# Build submission ZIP (combines both strategies)
zip_path = generate_submission(
    current_strategy=STRATEGY,
    results_dir=RESULTS_DIR,
    current_csv_path=current_csv_path,
    n_test=len(test_dataset),
)
print(f'\nDone! Upload to Codabench: {zip_path}')